In [ ]:
from google.colab import files
uploaded = files.upload()

Saving input_data.zip to input_data.zip


In [ ]:
import zipfile
import os

os.makedirs("data", exist_ok=True)

with zipfile.ZipFile("input_data.zip", "r") as zip_ref:
    zip_ref.extractall("data")

print(os.listdir("data"))

['train.json', 'test.json', '.DS_Store']


In [ ]:
import json

test_data = json.load(open("data/test.json"))

print("Total test samples:", len(test_data))
print(test_data[0])

Total test samples: 472
{'id': 'test_00001', 'dialogue_id': 'ESConv_001', 'dialogue': [{'text': 'hi', 'speaker': 'seeker'}, {'text': "Hi. How's it goin'? Are you feeling troubled about something this morning? I know it can be hard to talk about personal problems sometimes. I have a lot of trouble with that myself.", 'speaker': 'supporter'}, {'text': 'going smooth, i need to get a nice and a surprise able present for my parent, and at the same time I need to pay my house bills my father understands the situation that my earning is not that much, but my mum will fuck complain that I am taking care of my wife but not her', 'speaker': 'seeker'}, {'text': "It sounds like you're having some cash flow issues, right? I can relate to that at a few more points in my life than I'd points in my life than I'd like to recall. It's so very loving and sweet of you to be putting so much thought into the gift! That isn't all that common anymore, unfortunately.. Are you currently working?", 'speaker': 's

In [ ]:
import json

test_data = json.load(open("data/test.json"))

predictions = [
    {"id": x["id"], "label": 0}
    for x in test_data
]

with open("prediction.json", "w") as f:
    json.dump(predictions, f, indent=2)

print("Created prediction.json with", len(predictions), "entries")
print(predictions[:3])

Created prediction.json with 472 entries
[{'id': 'test_00001', 'label': 0}, {'id': 'test_00002', 'label': 0}, {'id': 'test_00003', 'label': 0}]


In [ ]:
import zipfile

with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write("prediction.json", arcname="prediction.json")

print("submission.zip created")

submission.zip created


In [ ]:
with zipfile.ZipFile("submission.zip", "r") as zipf:
    print(zipf.namelist())

['prediction.json']


In [ ]:
from google.colab import files
files.download("submission.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =========================
# PsyDefDetect Colab Pipeline
# Fixed for newer Transformers
# =========================

# 1) Install dependencies
!pip -q install transformers datasets evaluate scikit-learn accelerate

# 2) Imports
import os
import json
import zipfile
import random
import numpy as np
from collections import Counter

import torch
from torch import nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed
)

# 3) Config
set_seed(42)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

TRAIN_PATH = "data/train.json"
TEST_PATH = "data/test.json"

MODEL_NAME = "roberta-base"
MAX_LEN = 256
NUM_LABELS = 9

OUTPUT_DIR = "./psydef_roberta_output"
SUBMISSION_JSON = "prediction.json"
SUBMISSION_ZIP = "submission.zip"

# 4) Load data
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Train size:", len(train_data))
print("Test size:", len(test_data))

train_labels_all = [x["label"] for x in train_data]
print("Label distribution:", Counter(train_labels_all))

# 5) Build model input text
def build_input(example, max_turns=6):
    dialogue = example["dialogue"][-max_turns:]
    parts = []

    for turn in dialogue:
        speaker = str(turn["speaker"]).strip().lower()
        text = str(turn["text"]).strip()
        parts.append(f"{speaker}: {text}")

    context = "\n".join(parts)
    target = str(example["current_text"]).strip()

    full_text = (
        "[CONTEXT]\n"
        f"{context}\n\n"
        "[TARGET]\n"
        f"{target}"
    )
    return full_text

# 6) Train/validation split
train_split, valid_split = train_test_split(
    train_data,
    test_size=0.15,
    random_state=42,
    stratify=[x["label"] for x in train_data]
)

print("Train split:", len(train_split))
print("Valid split:", len(valid_split))

# 7) Convert to HF datasets
train_texts = [build_input(x) for x in train_split]
train_labels = [x["label"] for x in train_split]

valid_texts = [build_input(x) for x in valid_split]
valid_labels = [x["label"] for x in valid_split]

test_texts = [build_input(x) for x in test_data]

train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels})
valid_ds = Dataset.from_dict({"text": valid_texts, "label": valid_labels})
test_ds = Dataset.from_dict({"text": test_texts})

# 8) Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
valid_ds = valid_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

# rename label column to labels for Trainer compatibility
train_ds = train_ds.rename_column("label", "labels")
valid_ds = valid_ds.rename_column("label", "labels")

# optional, cleaner dataset format
train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
valid_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 9) Compute class weights
classes = np.array(sorted(list(set(train_labels))))
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=np.array(train_labels)
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

print("Classes:", classes)
print("Class weights:", class_weights)

# 10) Custom trainer with weighted cross-entropy
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device)
        )
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# 11) Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted")
    }

# 12) Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

# 13) Training args
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available()
)

# 14) Trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# 15) Train
trainer.train()

# 16) Validation results
pred_output = trainer.predict(valid_ds)
valid_preds = np.argmax(pred_output.predictions, axis=1)

print("\nValidation Accuracy:", accuracy_score(valid_labels, valid_preds))
print("Validation Macro-F1:", f1_score(valid_labels, valid_preds, average="macro"))
print("Validation Weighted-F1:", f1_score(valid_labels, valid_preds, average="weighted"))
print("\nClassification Report:\n")
print(classification_report(valid_labels, valid_preds, digits=4))

# 17) Predict test set
test_output = trainer.predict(test_ds)
test_preds = np.argmax(test_output.predictions, axis=1)

print("Number of test predictions:", len(test_preds))
print("First 20 predictions:", test_preds[:20])

# 18) Create prediction.json
submission = [
    {"id": ex["id"], "label": int(pred)}
    for ex, pred in zip(test_data, test_preds)
]

with open(SUBMISSION_JSON, "w", encoding="utf-8") as f:
    json.dump(submission, f, indent=2, ensure_ascii=False)

print(f"{SUBMISSION_JSON} created.")
print("First 5 submission rows:", submission[:5])

# 19) Sanity checks
loaded_pred = json.load(open(SUBMISSION_JSON, "r", encoding="utf-8"))
print("Submission length:", len(loaded_pred))
print("Matches test size:", len(loaded_pred) == len(test_data))
print("First row:", loaded_pred[0])
print("Last row:", loaded_pred[-1])

# 20) Create submission.zip with prediction.json at root
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(SUBMISSION_JSON, arcname="prediction.json")

print(f"{SUBMISSION_ZIP} created.")

with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zipf:
    print("ZIP contents:", zipf.namelist())

# 21) Download submission.zip
from google.colab import files
files.download(SUBMISSION_ZIP)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
Train size: 1864
Test size: 472
Label distribution: Counter({7: 968, 0: 296, 6: 172, 1: 108, 3: 99, 4: 84, 2: 61, 5: 48, 8: 28})
Train split: 1584
Valid split: 280


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Validation Accuracy: 0.525
Validation Macro-F1: 0.3034298849828673
Validation Weighted-F1: 0.5464191130359581

Classification Report:

              precision    recall  f1-score   support

           0     0.7593    0.9111    0.8283        45
           1     0.1538    0.1250    0.1379        16
           2     0.2000    0.4444    0.2759         9
           3     0.5000    0.1333    0.2105        15
           4     0.0882    0.2308    0.1277        13
           5     0.1429    0.1429    0.1429         7
           6     0.2895    0.4231    0.3438        26
           7     0.7905    0.5724    0.6640       145
           8     0.0000    0.0000    0.0000         4

    accuracy                         0.5250       280
   macro avg     0.3249    0.3314    0.3034       280
weighted avg     0.6079    0.5250    0.5464       280



Number of test predictions: 472
First 20 predictions: [6 7 0 7 1 0 3 0 6 1 1 7 7 0 0 0 7 7 7 7]
prediction.json created.
First 5 submission rows: [{'id': 'test_00001', 'label': 6}, {'id': 'test_00002', 'label': 7}, {'id': 'test_00003', 'label': 0}, {'id': 'test_00004', 'label': 7}, {'id': 'test_00005', 'label': 1}]
Submission length: 472
Matches test size: True
First row: {'id': 'test_00001', 'label': 6}
Last row: {'id': 'test_00472', 'label': 4}
submission.zip created.
ZIP contents: ['prediction.json']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Roberta Modified

In [ ]:
# =========================
# PsyDefDetect Colab Pipeline
# Safer improved RoBERTa version
# Changes:
# - MAX_LEN increased to 320
# - learning_rate reduced to 1.5e-5
# - warmup_steps added
# - fp16 made safer
# =========================

# 1) Install dependencies
!pip -q install transformers datasets evaluate scikit-learn accelerate

# 2) Imports
import json
import zipfile
import random
import numpy as np
from collections import Counter

import torch
from torch import nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed
)

# 3) Config
set_seed(42)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

TRAIN_PATH = "data/train.json"
TEST_PATH = "data/test.json"

MODEL_NAME = "roberta-base"
MAX_LEN = 320
NUM_LABELS = 9

OUTPUT_DIR = "./psydef_roberta_output_v2"
SUBMISSION_JSON = "prediction.json"
SUBMISSION_ZIP = "submission.zip"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 4) Load data
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Train size:", len(train_data))
print("Test size:", len(test_data))

train_labels_all = [x["label"] for x in train_data]
print("Label distribution:", Counter(train_labels_all))

# 5) Build model input text
def build_input(example, max_turns=6):
    dialogue = example["dialogue"][-max_turns:]
    parts = []

    for turn in dialogue:
        speaker = str(turn["speaker"]).strip().lower()
        text = str(turn["text"]).strip()
        parts.append(f"{speaker}: {text}")

    context = "\n".join(parts)
    target = str(example["current_text"]).strip()

    full_text = (
        "[CONTEXT]\n"
        f"{context}\n\n"
        "[TARGET]\n"
        f"{target}"
    )
    return full_text

# 6) Train/validation split
train_split, valid_split = train_test_split(
    train_data,
    test_size=0.15,
    random_state=42,
    stratify=[x["label"] for x in train_data]
)

print("Train split:", len(train_split))
print("Valid split:", len(valid_split))
print("Valid label distribution:", Counter(x["label"] for x in valid_split))

# 7) Convert to HF datasets
train_texts = [build_input(x) for x in train_split]
train_labels = [x["label"] for x in train_split]

valid_texts = [build_input(x) for x in valid_split]
valid_labels = [x["label"] for x in valid_split]

test_texts = [build_input(x) for x in test_data]

train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels})
valid_ds = Dataset.from_dict({"text": valid_texts, "label": valid_labels})
test_ds = Dataset.from_dict({"text": test_texts})

# 8) Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
valid_ds = valid_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

train_ds = train_ds.rename_column("label", "labels")
valid_ds = valid_ds.rename_column("label", "labels")

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
valid_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 9) Compute class weights
classes = np.array(sorted(list(set(train_labels))))
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=np.array(train_labels)
)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

print("Classes:", classes)
print("Class weights:", class_weights)

# 10) Custom trainer with weighted cross-entropy
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device).type_as(logits)
        )
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# 11) Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted")
    }

# 12) Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

# 13) Training args
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=1.5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="linear",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    bf16=False,
    max_grad_norm=1.0,
    dataloader_pin_memory=torch.cuda.is_available()
)

# 14) Trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# 15) Train
trainer.train()

# 16) Validation results
pred_output = trainer.predict(valid_ds)
valid_preds = np.argmax(pred_output.predictions, axis=1)

print("\nValidation Accuracy:", accuracy_score(valid_labels, valid_preds))
print("Validation Macro-F1:", f1_score(valid_labels, valid_preds, average="macro"))
print("Validation Weighted-F1:", f1_score(valid_labels, valid_preds, average="weighted"))
print("\nClassification Report:\n")
print(classification_report(valid_labels, valid_preds, digits=4))

# 17) Predict test set
test_output = trainer.predict(test_ds)
test_preds = np.argmax(test_output.predictions, axis=1)

print("Number of test predictions:", len(test_preds))
print("First 20 predictions:", test_preds[:20])

# 18) Create prediction.json
submission = [
    {"id": ex["id"], "label": int(pred)}
    for ex, pred in zip(test_data, test_preds)
]

with open(SUBMISSION_JSON, "w", encoding="utf-8") as f:
    json.dump(submission, f, indent=2, ensure_ascii=False)

print(f"{SUBMISSION_JSON} created.")
print("First 5 submission rows:", submission[:5])

# 19) Sanity checks
loaded_pred = json.load(open(SUBMISSION_JSON, "r", encoding="utf-8"))
print("Submission length:", len(loaded_pred))
print("Matches test size:", len(loaded_pred) == len(test_data))
print("First row:", loaded_pred[0])
print("Last row:", loaded_pred[-1])

# 20) Create submission.zip with prediction.json at root
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(SUBMISSION_JSON, arcname="prediction.json")

print(f"{SUBMISSION_ZIP} created.")

with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zipf:
    print("ZIP contents:", zipf.namelist())

# 21) Download submission.zip
from google.colab import files
files.download(SUBMISSION_ZIP)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
CUDA available: True
GPU: Tesla T4
Train size: 1864
Test size: 472
Label distribution: Counter({7: 968, 0: 296, 6: 172, 1: 108, 3: 99, 4: 84, 2: 61, 5: 48, 8: 28})
Train split: 1584
Valid split: 280
Valid label distribution: Counter({7: 145, 0: 45, 6: 26, 1: 16, 3: 15, 4: 13, 2: 9, 5: 7, 8: 4})


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Validation Accuracy: 0.4785714285714286
Validation Macro-F1: 0.27631820317595157
Validation Weighted-F1: 0.4900393647074572

Classification Report:

              precision    recall  f1-score   support

           0     0.6324    0.9556    0.7611        45
           1     0.2000    0.2500    0.2222        16
           2     0.1818    0.6667    0.2857         9
           3     0.0000    0.0000    0.0000        15
           4     0.1875    0.2308    0.2069        13
           5     0.1667    0.1429    0.1538         7
           6     0.2286    0.3077    0.2623        26
           7     0.7931    0.4759    0.5948       145
           8     0.0000    0.0000    0.0000         4

    accuracy                         0.4786       280
   macro avg     0.2656    0.3366    0.2763       280
weighted avg     0.5637    0.4786    0.4900       280



Number of test predictions: 472
First 20 predictions: [6 7 0 6 1 0 8 0 1 6 1 7 7 0 0 0 7 7 7 7]
prediction.json created.
First 5 submission rows: [{'id': 'test_00001', 'label': 6}, {'id': 'test_00002', 'label': 7}, {'id': 'test_00003', 'label': 0}, {'id': 'test_00004', 'label': 6}, {'id': 'test_00005', 'label': 1}]
Submission length: 472
Matches test size: True
First row: {'id': 'test_00001', 'label': 6}
Last row: {'id': 'test_00472', 'label': 1}
submission.zip created.
ZIP contents: ['prediction.json']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Roberta with oversampling

In [ ]:
# =========================
# PsyDefDetect RoBERTa + Random Oversampling
# Goal:
# - Improve minority classes safely
# - Oversample only training data
# - Keep validation untouched
# - Create prediction.json + submission.zip
# =========================

# 1) Install dependencies
!pip -q install transformers datasets evaluate scikit-learn accelerate

# 2) Imports
import json
import zipfile
import random
import numpy as np
from collections import Counter

import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed
)

# 3) Config
set_seed(42)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

TRAIN_PATH = "data/train.json"
TEST_PATH = "data/test.json"

MODEL_NAME = "roberta-base"
MAX_LEN = 256
NUM_LABELS = 9

OUTPUT_DIR = "./psydef_roberta_oversampled"
SUBMISSION_JSON = "prediction.json"
SUBMISSION_ZIP = "submission.zip"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 4) Load data
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Original train size:", len(train_data))
print("Test size:", len(test_data))
print("Original label distribution:", Counter(x["label"] for x in train_data))

# 5) Build model input text
def build_input(example, max_turns=6):
    dialogue = example["dialogue"][-max_turns:]
    parts = []

    for turn in dialogue:
        speaker = str(turn["speaker"]).strip().lower()
        text = str(turn["text"]).strip()
        parts.append(f"{speaker}: {text}")

    context = "\n".join(parts)
    target = str(example["current_text"]).strip()

    full_text = (
        "[CONTEXT]\n"
        f"{context}\n\n"
        "[TARGET]\n"
        f"{target}"
    )
    return full_text

# 6) Train/validation split
train_split, valid_split = train_test_split(
    train_data,
    test_size=0.15,
    random_state=42,
    stratify=[x["label"] for x in train_data]
)

print("Train split before oversampling:", len(train_split))
print("Valid split:", len(valid_split))
print("Train label distribution before oversampling:", Counter(x["label"] for x in train_split))
print("Valid label distribution:", Counter(x["label"] for x in valid_split))

# 7) Random oversampling on training split only
# Strategy:
# - Bring smaller classes up to a moderate target size
# - Do NOT make every class equal to class 7
# - Keep validation untouched

def oversample_training_data(train_examples, target_min_count=120):
    by_label = {}
    for ex in train_examples:
        lbl = ex["label"]
        by_label.setdefault(lbl, []).append(ex)

    oversampled = []
    for lbl, examples in sorted(by_label.items()):
        current_count = len(examples)

        # keep large classes as they are
        if current_count >= target_min_count:
            oversampled.extend(examples)
        else:
            extra_needed = target_min_count - current_count
            sampled_extra = random.choices(examples, k=extra_needed)
            oversampled.extend(examples + sampled_extra)

    random.shuffle(oversampled)
    return oversampled

train_split_os = oversample_training_data(train_split, target_min_count=120)

print("Train split after oversampling:", len(train_split_os))
print("Train label distribution after oversampling:", Counter(x["label"] for x in train_split_os))

# 8) Convert to HF datasets
train_texts = [build_input(x) for x in train_split_os]
train_labels = [x["label"] for x in train_split_os]

valid_texts = [build_input(x) for x in valid_split]
valid_labels = [x["label"] for x in valid_split]

test_texts = [build_input(x) for x in test_data]

train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels})
valid_ds = Dataset.from_dict({"text": valid_texts, "label": valid_labels})
test_ds = Dataset.from_dict({"text": test_texts})

# 9) Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
valid_ds = valid_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

train_ds = train_ds.rename_column("label", "labels")
valid_ds = valid_ds.rename_column("label", "labels")

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
valid_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 10) Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted")
    }

# 11) Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

# 12) Training args
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="linear",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    bf16=False,
    max_grad_norm=1.0,
    dataloader_pin_memory=torch.cuda.is_available()
)

# 13) Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# 14) Train
trainer.train()

# 15) Validation results
pred_output = trainer.predict(valid_ds)
valid_preds = np.argmax(pred_output.predictions, axis=1)

print("\nValidation Accuracy:", accuracy_score(valid_labels, valid_preds))
print("Validation Macro-F1:", f1_score(valid_labels, valid_preds, average="macro"))
print("Validation Weighted-F1:", f1_score(valid_labels, valid_preds, average="weighted"))
print("\nClassification Report:\n")
print(classification_report(valid_labels, valid_preds, digits=4))

# 16) Predict test set
test_output = trainer.predict(test_ds)
test_preds = np.argmax(test_output.predictions, axis=1)

print("Number of test predictions:", len(test_preds))
print("First 20 predictions:", test_preds[:20])

# 17) Create prediction.json
submission = [
    {"id": ex["id"], "label": int(pred)}
    for ex, pred in zip(test_data, test_preds)
]

with open(SUBMISSION_JSON, "w", encoding="utf-8") as f:
    json.dump(submission, f, indent=2, ensure_ascii=False)

print(f"{SUBMISSION_JSON} created.")
print("First 5 submission rows:", submission[:5])

# 18) Sanity checks
loaded_pred = json.load(open(SUBMISSION_JSON, "r", encoding="utf-8"))
print("Submission length:", len(loaded_pred))
print("Matches test size:", len(loaded_pred) == len(test_data))
print("First row:", loaded_pred[0])
print("Last row:", loaded_pred[-1])

# 19) Create submission.zip with prediction.json at root
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(SUBMISSION_JSON, arcname="prediction.json")

print(f"{SUBMISSION_ZIP} created.")
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zipf:
    print("ZIP contents:", zipf.namelist())

# 20) Download submission.zip
from google.colab import files
files.download(SUBMISSION_ZIP)

CUDA available: True
GPU: Tesla T4
Original train size: 1864
Test size: 472
Original label distribution: Counter({7: 968, 0: 296, 6: 172, 1: 108, 3: 99, 4: 84, 2: 61, 5: 48, 8: 28})
Train split before oversampling: 1584
Valid split: 280
Train label distribution before oversampling: Counter({7: 823, 0: 251, 6: 146, 1: 92, 3: 84, 4: 71, 2: 52, 5: 41, 8: 24})
Valid label distribution: Counter({7: 145, 0: 45, 6: 26, 1: 16, 3: 15, 4: 13, 2: 9, 5: 7, 8: 4})
Train split after oversampling: 1940
Train label distribution after oversampling: Counter({7: 823, 0: 251, 6: 146, 8: 120, 5: 120, 2: 120, 4: 120, 1: 120, 3: 120})


Map:   0%|          | 0/1940 [00:00<?, ? examples/s]

Map:   0%|          | 0/280 [00:00<?, ? examples/s]

Map:   0%|          | 0/472 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Validation Accuracy: 0.6178571428571429
Validation Macro-F1: 0.2893066736066006
Validation Weighted-F1: 0.5763823364372774

Classification Report:

              precision    recall  f1-score   support

           0     0.7917    0.8444    0.8172        45
           1     0.1667    0.1250    0.1429        16
           2     0.5714    0.4444    0.5000         9
           3     0.0000    0.0000    0.0000        15
           4     0.2000    0.1538    0.1739        13
           5     0.0000    0.0000    0.0000         7
           6     0.3333    0.1538    0.2105        26
           7     0.6872    0.8483    0.7593       145
           8     0.0000    0.0000    0.0000         4

    accuracy                         0.6179       280
   macro avg     0.3056    0.2855    0.2893       280
weighted avg     0.5512    0.6179    0.5764       280



Number of test predictions: 472
First 20 predictions: [6 7 0 7 7 5 8 0 7 7 1 7 7 0 0 0 7 7 7 1]
prediction.json created.
First 5 submission rows: [{'id': 'test_00001', 'label': 6}, {'id': 'test_00002', 'label': 7}, {'id': 'test_00003', 'label': 0}, {'id': 'test_00004', 'label': 7}, {'id': 'test_00005', 'label': 7}]
Submission length: 472
Matches test size: True
First row: {'id': 'test_00001', 'label': 6}
Last row: {'id': 'test_00472', 'label': 7}
submission.zip created.
ZIP contents: ['prediction.json']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 2nd oversampling

In [ ]:
# =========================
# PsyDefDetect RoBERTa + Random Oversampling
# Goal:
# - Improve minority classes safely
# - Oversample only training data
# - Keep validation untouched
# - Create prediction.json + submission.zip
# =========================

# 1) Install dependencies
!pip -q install transformers datasets evaluate scikit-learn accelerate

# 2) Imports
import json
import zipfile
import random
import numpy as np
from collections import Counter

import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed
)

# 3) Config
set_seed(42)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

TRAIN_PATH = "data/train.json"
TEST_PATH = "data/test.json"

MODEL_NAME = "roberta-base"
MAX_LEN = 256
NUM_LABELS = 9

OUTPUT_DIR = "./psydef_roberta_oversampled"
SUBMISSION_JSON = "prediction.json"
SUBMISSION_ZIP = "submission.zip"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 4) Load data
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Original train size:", len(train_data))
print("Test size:", len(test_data))
print("Original label distribution:", Counter(x["label"] for x in train_data))

# 5) Build model input text
def build_input(example, max_turns=6):
    dialogue = example["dialogue"][-max_turns:]
    parts = []

    for turn in dialogue:
        speaker = str(turn["speaker"]).strip().lower()
        text = str(turn["text"]).strip()
        parts.append(f"{speaker}: {text}")

    context = "\n".join(parts)
    target = str(example["current_text"]).strip()

    full_text = (
        "[CONTEXT]\n"
        f"{context}\n\n"
        "[TARGET]\n"
        f"{target}"
    )
    return full_text

# 6) Train/validation split
train_split, valid_split = train_test_split(
    train_data,
    test_size=0.15,
    random_state=42,
    stratify=[x["label"] for x in train_data]
)

print("Train split before oversampling:", len(train_split))
print("Valid split:", len(valid_split))
print("Train label distribution before oversampling:", Counter(x["label"] for x in train_split))
print("Valid label distribution:", Counter(x["label"] for x in valid_split))

# 7) Random oversampling on training split only
# Strategy:
# - Bring smaller classes up to a moderate target size
# - Do NOT make every class equal to class 7
# - Keep validation untouched

# -------------------------
# Targeted oversampling (NEW)
# -------------------------
def oversample_targeted(train_examples):
    by_label = {}
    for ex in train_examples:
        lbl = ex["label"]
        by_label.setdefault(lbl, []).append(ex)

    oversampled = []

    for lbl, examples in by_label.items():
        count = len(examples)

        if lbl == 8:
            target = 180   # ↑ from 150
        elif lbl == 5:
            target = 160
        elif lbl == 3:
            target = 150
        else:
            target = count  # keep others unchanged

        if count < target:
            extra = random.choices(examples, k=target - count)
            oversampled.extend(examples + extra)
        else:
            oversampled.extend(examples)

    random.shuffle(oversampled)
    return oversampled

train_split_os = oversample_targeted(train_split)

print("Train split after oversampling:", len(train_split_os))
print("Train label distribution after oversampling:", Counter(x["label"] for x in train_split_os))

# 8) Convert to HF datasets
train_texts = [build_input(x) for x in train_split_os]
train_labels = [x["label"] for x in train_split_os]

valid_texts = [build_input(x) for x in valid_split]
valid_labels = [x["label"] for x in valid_split]

test_texts = [build_input(x) for x in test_data]

train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels})
valid_ds = Dataset.from_dict({"text": valid_texts, "label": valid_labels})
test_ds = Dataset.from_dict({"text": test_texts})

# 9) Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
valid_ds = valid_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

train_ds = train_ds.rename_column("label", "labels")
valid_ds = valid_ds.rename_column("label", "labels")

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
valid_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 10) Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted")
    }

# 11) Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

# 12) Training args
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="linear",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    bf16=False,
    max_grad_norm=1.0,
    dataloader_pin_memory=torch.cuda.is_available()
)

# 13) Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# 14) Train
trainer.train()

# 15) Validation results
pred_output = trainer.predict(valid_ds)
valid_preds = np.argmax(pred_output.predictions, axis=1)

print("\nValidation Accuracy:", accuracy_score(valid_labels, valid_preds))
print("Validation Macro-F1:", f1_score(valid_labels, valid_preds, average="macro"))
print("Validation Weighted-F1:", f1_score(valid_labels, valid_preds, average="weighted"))
print("\nClassification Report:\n")
print(classification_report(valid_labels, valid_preds, digits=4))

# 16) Predict test set
test_output = trainer.predict(test_ds)
test_preds = np.argmax(test_output.predictions, axis=1)

print("Number of test predictions:", len(test_preds))
print("First 20 predictions:", test_preds[:20])

# 17) Create prediction.json
submission = [
    {"id": ex["id"], "label": int(pred)}
    for ex, pred in zip(test_data, test_preds)
]

with open(SUBMISSION_JSON, "w", encoding="utf-8") as f:
    json.dump(submission, f, indent=2, ensure_ascii=False)

print(f"{SUBMISSION_JSON} created.")
print("First 5 submission rows:", submission[:5])

# 18) Sanity checks
loaded_pred = json.load(open(SUBMISSION_JSON, "r", encoding="utf-8"))
print("Submission length:", len(loaded_pred))
print("Matches test size:", len(loaded_pred) == len(test_data))
print("First row:", loaded_pred[0])
print("Last row:", loaded_pred[-1])

# 19) Create submission.zip with prediction.json at root
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(SUBMISSION_JSON, arcname="prediction.json")

print(f"{SUBMISSION_ZIP} created.")
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zipf:
    print("ZIP contents:", zipf.namelist())

# 20) Download submission.zip
from google.colab import files
files.download(SUBMISSION_ZIP)

CUDA available: True
GPU: Tesla T4
Original train size: 1864
Test size: 472
Original label distribution: Counter({7: 968, 0: 296, 6: 172, 1: 108, 3: 99, 4: 84, 2: 61, 5: 48, 8: 28})
Train split before oversampling: 1584
Valid split: 280
Train label distribution before oversampling: Counter({7: 823, 0: 251, 6: 146, 1: 92, 3: 84, 4: 71, 2: 52, 5: 41, 8: 24})
Valid label distribution: Counter({7: 145, 0: 45, 6: 26, 1: 16, 3: 15, 4: 13, 2: 9, 5: 7, 8: 4})
Train split after oversampling: 1925
Train label distribution after oversampling: Counter({7: 823, 0: 251, 8: 180, 5: 160, 3: 150, 6: 146, 1: 92, 4: 71, 2: 52})


Map:   0%|          | 0/1925 [00:00<?, ? examples/s]

Map:   0%|          | 0/280 [00:00<?, ? examples/s]

Map:   0%|          | 0/472 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Validation Accuracy: 0.6142857142857143
Validation Macro-F1: 0.243031431665959
Validation Weighted-F1: 0.574988291401569

Classification Report:

              precision    recall  f1-score   support

           0     0.8163    0.8889    0.8511        45
           1     0.1667    0.0625    0.0909        16
           2     0.1667    0.1111    0.1333         9
           3     0.0000    0.0000    0.0000        15
           4     0.0000    0.0000    0.0000        13
           5     0.0000    0.0000    0.0000         7
           6     0.3462    0.3462    0.3462        26
           7     0.7076    0.8345    0.7658       145
           8     0.0000    0.0000    0.0000         4

    accuracy                         0.6143       280
   macro avg     0.2448    0.2492    0.2430       280
weighted avg     0.5447    0.6143    0.5750       280



Number of test predictions: 472
First 20 predictions: [6 7 0 7 7 5 6 0 7 7 6 7 7 0 0 0 7 7 7 7]
prediction.json created.
First 5 submission rows: [{'id': 'test_00001', 'label': 6}, {'id': 'test_00002', 'label': 7}, {'id': 'test_00003', 'label': 0}, {'id': 'test_00004', 'label': 7}, {'id': 'test_00005', 'label': 7}]
Submission length: 472
Matches test size: True
First row: {'id': 'test_00001', 'label': 6}
Last row: {'id': 'test_00472', 'label': 7}
submission.zip created.
ZIP contents: ['prediction.json']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =========================
# PsyDefDetect stronger pipeline
# Fixed full version:
# - DeBERTa-v3-base
# - focal loss
# - weighted sampler
# - dtype-safe class weights
# - creates prediction.json + submission.zip
# =========================

!pip -q install transformers datasets evaluate scikit-learn accelerate sentencepiece

import json
import zipfile
import random
import numpy as np
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed
)

# -------------------------
# 1. Config
# -------------------------
set_seed(42)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

TRAIN_PATH = "data/train.json"
TEST_PATH = "data/test.json"

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 384
NUM_LABELS = 9

OUTPUT_DIR = "./psydef_deberta_push"
SUBMISSION_JSON = "prediction.json"
SUBMISSION_ZIP = "submission.zip"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# -------------------------
# 2. Load data
# -------------------------
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Train size:", len(train_data))
print("Test size:", len(test_data))
print("Train label distribution:", Counter(x["label"] for x in train_data))

# -------------------------
# 3. Better input builder
# -------------------------
def build_input(example, max_turns=6):
    dialogue = example["dialogue"][-max_turns:]
    turns = []

    for t in dialogue:
        speaker = str(t["speaker"]).strip().lower()
        text = str(t["text"]).strip()
        turns.append(f"{speaker}: {text}")

    context = " </s> ".join(turns)
    target = str(example["current_text"]).strip()

    full_text = (
        f"Dialogue context: {context} </s> "
        f"Target utterance: {target} </s> "
        f"Task: predict the psychological defense mechanism tier from 0 to 8."
    )
    return full_text

# -------------------------
# 4. Split data
# -------------------------
train_split, valid_split = train_test_split(
    train_data,
    test_size=0.15,
    random_state=42,
    stratify=[x["label"] for x in train_data]
)

print("Train split:", len(train_split))
print("Valid split:", len(valid_split))
print("Valid label distribution:", Counter(x["label"] for x in valid_split))

train_texts = [build_input(x) for x in train_split]
train_labels = [x["label"] for x in train_split]

valid_texts = [build_input(x) for x in valid_split]
valid_labels = [x["label"] for x in valid_split]

test_texts = [build_input(x) for x in test_data]

train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels})
valid_ds = Dataset.from_dict({"text": valid_texts, "label": valid_labels})
test_ds = Dataset.from_dict({"text": test_texts})

# -------------------------
# 5. Tokenizer
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
valid_ds = valid_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

train_ds = train_ds.rename_column("label", "labels")
valid_ds = valid_ds.rename_column("label", "labels")

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
valid_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# -------------------------
# 6. Class weights
# -------------------------
classes = np.array(sorted(list(set(train_labels))))
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=np.array(train_labels)
)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

print("Classes:", classes)
print("Class weights:", class_weights)

# -------------------------
# 7. Weighted sampler
# -------------------------
label_counts = Counter(train_labels)
sample_weights = [1.0 / label_counts[label] for label in train_labels]
sample_weights = torch.DoubleTensor(sample_weights)

weighted_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# -------------------------
# 8. Focal loss
# -------------------------
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        # Match alpha dtype/device with logits to avoid Half/Float mismatch
        if self.alpha is not None:
            alpha = self.alpha.to(logits.device).type_as(logits)
        else:
            alpha = None

        ce_loss = nn.functional.cross_entropy(
            logits,
            targets,
            reduction="none",
            weight=alpha
        )
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# -------------------------
# 9. Custom trainer
# -------------------------
class CustomTrainer(Trainer):
    def __init__(self, class_weights=None, sampler=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.sampler = sampler
        self.focal_loss = FocalLoss(alpha=class_weights, gamma=2.0)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = self.focal_loss(logits, labels)
        return (loss, outputs) if return_outputs else loss

    def get_train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.per_device_train_batch_size,
            sampler=self.sampler,
            collate_fn=self.data_collator,
            num_workers=2,
            pin_memory=torch.cuda.is_available()
        )

# -------------------------
# 10. Metrics
# -------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted")
    }

# -------------------------
# 11. Model
# -------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

# -------------------------
# 12. Training args
# -------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=1.5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=6,
    weight_decay=0.02,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=False,
    bf16=False,
    dataloader_pin_memory=torch.cuda.is_available(),
    max_grad_norm=1.0
)

# -------------------------
# 13. Trainer
# -------------------------
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    sampler=weighted_sampler,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# -------------------------
# 14. Train
# -------------------------
trainer.train()

# -------------------------
# 15. Validate
# -------------------------
val_output = trainer.predict(valid_ds)
val_preds = np.argmax(val_output.predictions, axis=1)

print("\nValidation Accuracy:", accuracy_score(valid_labels, val_preds))
print("Validation Macro-F1:", f1_score(valid_labels, val_preds, average="macro"))
print("Validation Weighted-F1:", f1_score(valid_labels, val_preds, average="weighted"))
print("\nClassification Report:\n")
print(classification_report(valid_labels, val_preds, digits=4))

# -------------------------
# 16. Predict test
# -------------------------
test_output = trainer.predict(test_ds)
test_logits = test_output.predictions
test_preds = np.argmax(test_logits, axis=1)

print("Number of test predictions:", len(test_preds))
print("First 20 predictions:", test_preds[:20])

# -------------------------
# 17. Create prediction.json
# -------------------------
submission = [
    {"id": ex["id"], "label": int(pred)}
    for ex, pred in zip(test_data, test_preds)
]

with open(SUBMISSION_JSON, "w", encoding="utf-8") as f:
    json.dump(submission, f, indent=2, ensure_ascii=False)

print(f"{SUBMISSION_JSON} created")
print(submission[:5])

# -------------------------
# 18. Zip for submission
# -------------------------
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(SUBMISSION_JSON, arcname="prediction.json")

print(f"{SUBMISSION_ZIP} created")
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zipf:
    print("ZIP contents:", zipf.namelist())

# -------------------------
# 19. Download
# -------------------------
from google.colab import files
files.download(SUBMISSION_ZIP)

CUDA available: True
GPU: Tesla T4
Train size: 1864
Test size: 472
Train label distribution: Counter({7: 968, 0: 296, 6: 172, 1: 108, 3: 99, 4: 84, 2: 61, 5: 48, 8: 28})
Train split: 1584
Valid split: 280
Valid label distribution: Counter({7: 145, 0: 45, 6: 26, 1: 16, 3: 15, 4: 13, 2: 9, 5: 7, 8: 4})


Map:   0%|          | 0/1584 [00:00<?, ? examples/s]

Map:   0%|          | 0/280 [00:00<?, ? examples/s]

Map:   0%|          | 0/472 [00:00<?, ? examples/s]

Classes: [0 1 2 3 4 5 6 7 8]
Class weights: tensor([0.7012, 1.9130, 3.3846, 2.0952, 2.4789, 4.2927, 1.2055, 0.2139, 7.3333])


...


Validation Accuracy: 0.014285714285714285
Validation Macro-F1: 0.003129890453834116
Validation Weighted-F1: 0.00040241448692152917

Classification Report:

              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000        45
           1     0.0000    0.0000    0.0000        16
           2     0.0000    0.0000    0.0000         9
           3     0.0000    0.0000    0.0000        15
           4     0.0000    0.0000    0.0000        13
           5     0.0000    0.0000    0.0000         7
           6     0.0000    0.0000    0.0000        26
           7     0.0000    0.0000    0.0000       145
           8     0.0143    1.0000    0.0282         4

    accuracy                         0.0143       280
   macro avg     0.0016    0.1111    0.0031       280
weighted avg     0.0002    0.0143    0.0004       280



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Number of test predictions: 472
First 20 predictions: [8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8]
prediction.json created
[{'id': 'test_00001', 'label': 8}, {'id': 'test_00002', 'label': 8}, {'id': 'test_00003', 'label': 8}, {'id': 'test_00004', 'label': 8}, {'id': 'test_00005', 'label': 8}]
submission.zip created
ZIP contents: ['prediction.json']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =========================
# PsyDefDetect clean DeBERTa pipeline
# Stable version:
# - No focal loss
# - No weighted sampler
# - No custom trainer
# - Uses default Trainer
# - Creates prediction.json + submission.zip
# =========================

!pip -q install transformers datasets evaluate scikit-learn accelerate sentencepiece

import json
import zipfile
import random
import numpy as np
from collections import Counter

import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed
)

# -------------------------
# 1. Config
# -------------------------
set_seed(42)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

TRAIN_PATH = "data/train.json"
TEST_PATH = "data/test.json"

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 256
NUM_LABELS = 9

OUTPUT_DIR = "./psydef_deberta_clean"
SUBMISSION_JSON = "prediction.json"
SUBMISSION_ZIP = "submission.zip"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# -------------------------
# 2. Load data
# -------------------------
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Train size:", len(train_data))
print("Test size:", len(test_data))
print("Train label distribution:", Counter(x["label"] for x in train_data))

# -------------------------
# 3. Build input text
# -------------------------
def build_input(example, max_turns=6):
    dialogue = example["dialogue"][-max_turns:]
    parts = []

    for turn in dialogue:
        speaker = str(turn["speaker"]).strip().lower()
        text = str(turn["text"]).strip()
        parts.append(f"{speaker}: {text}")

    context = " </s> ".join(parts)
    target = str(example["current_text"]).strip()

    return f"{context} </s> TARGET: {target}"

# -------------------------
# 4. Train/validation split
# -------------------------
train_split, valid_split = train_test_split(
    train_data,
    test_size=0.15,
    random_state=42,
    stratify=[x["label"] for x in train_data]
)

print("Train split:", len(train_split))
print("Valid split:", len(valid_split))
print("Valid label distribution:", Counter(x["label"] for x in valid_split))

# -------------------------
# 5. Convert to datasets
# -------------------------
train_texts = [build_input(x) for x in train_split]
train_labels = [x["label"] for x in train_split]

valid_texts = [build_input(x) for x in valid_split]
valid_labels = [x["label"] for x in valid_split]

test_texts = [build_input(x) for x in test_data]

train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels})
valid_ds = Dataset.from_dict({"text": valid_texts, "label": valid_labels})
test_ds = Dataset.from_dict({"text": test_texts})

# -------------------------
# 6. Tokenizer
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
valid_ds = valid_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

train_ds = train_ds.rename_column("label", "labels")
valid_ds = valid_ds.rename_column("label", "labels")

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
valid_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# -------------------------
# 7. Metrics
# -------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted")
    }

# -------------------------
# 8. Model
# -------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

# -------------------------
# 9. Training arguments
# -------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="linear",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=False,
    bf16=False,
    max_grad_norm=1.0,
    dataloader_pin_memory=torch.cuda.is_available()
)

# -------------------------
# 10. Trainer
# -------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# -------------------------
# 11. Train
# -------------------------
trainer.train()

# -------------------------
# 12. Validate
# -------------------------
val_output = trainer.predict(valid_ds)
val_preds = np.argmax(val_output.predictions, axis=1)

print("\nValidation Accuracy:", accuracy_score(valid_labels, val_preds))
print("Validation Macro-F1:", f1_score(valid_labels, val_preds, average="macro"))
print("Validation Weighted-F1:", f1_score(valid_labels, val_preds, average="weighted"))
print("\nClassification Report:\n")
print(classification_report(valid_labels, val_preds, digits=4))

# -------------------------
# 13. Predict test
# -------------------------
test_output = trainer.predict(test_ds)
test_preds = np.argmax(test_output.predictions, axis=1)

print("Number of test predictions:", len(test_preds))
print("First 20 predictions:", test_preds[:20])

# -------------------------
# 14. Create prediction.json
# -------------------------
submission = [
    {"id": ex["id"], "label": int(pred)}
    for ex, pred in zip(test_data, test_preds)
]

with open(SUBMISSION_JSON, "w", encoding="utf-8") as f:
    json.dump(submission, f, indent=2, ensure_ascii=False)

print(f"{SUBMISSION_JSON} created")
print("First 5 submission rows:", submission[:5])

# -------------------------
# 15. Sanity checks
# -------------------------
loaded_pred = json.load(open(SUBMISSION_JSON, "r", encoding="utf-8"))
print("Submission length:", len(loaded_pred))
print("Matches test size:", len(loaded_pred) == len(test_data))
print("First row:", loaded_pred[0])
print("Last row:", loaded_pred[-1])

# -------------------------
# 16. Zip for submission
# -------------------------
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(SUBMISSION_JSON, arcname="prediction.json")

print(f"{SUBMISSION_ZIP} created")
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zipf:
    print("ZIP contents:", zipf.namelist())

# -------------------------
# 17. Download
# -------------------------
from google.colab import files
files.download(SUBMISSION_ZIP)

CUDA available: True
GPU: Tesla T4
Train size: 1864
Test size: 472
Train label distribution: Counter({7: 968, 0: 296, 6: 172, 1: 108, 3: 99, 4: 84, 2: 61, 5: 48, 8: 28})
Train split: 1584
Valid split: 280
Valid label distribution: Counter({7: 145, 0: 45, 6: 26, 1: 16, 3: 15, 4: 13, 2: 9, 5: 7, 8: 4})


Map:   0%|          | 0/1584 [00:00<?, ? examples/s]

Map:   0%|          | 0/280 [00:00<?, ? examples/s]

Map:   0%|          | 0/472 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

...


Validation Accuracy: 0.16071428571428573
Validation Macro-F1: 0.03076923076923077
Validation Weighted-F1: 0.04450549450549451

Classification Report:

              precision    recall  f1-score   support

           0     0.1607    1.0000    0.2769        45
           1     0.0000    0.0000    0.0000        16
           2     0.0000    0.0000    0.0000         9
           3     0.0000    0.0000    0.0000        15
           4     0.0000    0.0000    0.0000        13
           5     0.0000    0.0000    0.0000         7
           6     0.0000    0.0000    0.0000        26
           7     0.0000    0.0000    0.0000       145
           8     0.0000    0.0000    0.0000         4

    accuracy                         0.1607       280
   macro avg     0.0179    0.1111    0.0308       280
weighted avg     0.0258    0.1607    0.0445       280



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Number of test predictions: 472
First 20 predictions: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
prediction.json created
First 5 submission rows: [{'id': 'test_00001', 'label': 0}, {'id': 'test_00002', 'label': 0}, {'id': 'test_00003', 'label': 0}, {'id': 'test_00004', 'label': 0}, {'id': 'test_00005', 'label': 0}]
Submission length: 472
Matches test size: True
First row: {'id': 'test_00001', 'label': 0}
Last row: {'id': 'test_00472', 'label': 0}
submission.zip created
ZIP contents: ['prediction.json']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

CUDA available: True
GPU: Tesla T4
Train size: 1864
Test size: 472
Train label distribution: Counter({7: 968, 0: 296, 6: 172, 1: 108, 3: 99, 4: 84, 2: 61, 5: 48, 8: 28})
Train split: 1584
Valid split: 280
Valid label distribution: Counter({7: 145, 0: 45, 6: 26, 1: 16, 3: 15, 4: 13, 2: 9, 5: 7, 8: 4})


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...


Validation Accuracy: 0.625
Validation Macro-F1: 0.1794838160398801
Validation Weighted-F1: 0.5175321270072414

Classification Report:

              precision    recall  f1-score   support

           0     0.7800    0.8667    0.8211        45
           1     0.0000    0.0000    0.0000        16
           2     0.0000    0.0000    0.0000         9
           3     0.0000    0.0000    0.0000        15
           4     0.0000    0.0000    0.0000        13
           5     0.0000    0.0000    0.0000         7
           6     0.1429    0.0385    0.0606        26
           7     0.6054    0.9310    0.7337       145
           8     0.0000    0.0000    0.0000         4

    accuracy                         0.6250       280
   macro avg     0.1698    0.2040    0.1795       280
weighted avg     0.4521    0.6250    0.5175       280



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Number of test predictions: 472
First 20 predictions: [7 7 0 7 7 7 7 0 7 7 7 7 7 0 0 0 7 7 7 7]
prediction.json created
First 5 submission rows: [{'id': 'test_00001', 'label': 7}, {'id': 'test_00002', 'label': 7}, {'id': 'test_00003', 'label': 0}, {'id': 'test_00004', 'label': 7}, {'id': 'test_00005', 'label': 7}]
Submission length: 472
Matches test size: True
First row: {'id': 'test_00001', 'label': 7}
Last row: {'id': 'test_00472', 'label': 7}
submission.zip created
ZIP contents: ['prediction.json']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>